LOADING ALL OS AND CALLING LIBRAIES

In [ ]:
import os
import random
from faker import Faker
import psycopg2

from dotenv import load_dotenv
from psycopg2.extras import execute_values
execute_values(cur, statement, records)

loaded = load_dotenv(r"C:\Users\User\OneDrive\Documents\MY 10ANALYTICS PROJECT\FinalCapstone\.env")
fake = Faker()

ENV ENVIRONMENT CONNECTION TO DB

In [19]:
host = os.getenv("DB_Host")
port = os.getenv("DB_PORT") 
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
database = os.getenv("DB_NAME")

CONNECTION TO DATABASE

In [20]:
conn = psycopg2.connect(
    host=host,
    database=database,
    user=user,
    password=password,
    port=port
)

cur = conn.cursor()
print("Connected successfully")

Connected successfully


In [21]:
conn.rollback()

TABLE STORE POPULATED

In [ ]:
# Define the number of records
number_records = 5

# instantiate Faker object
fake = Faker()

records = []

# Generate and insert data into the store table
for _ in range(number_records):
    address = fake.address()
    city = fake.city()
    phone_number = fake.phone_number()
    opened_at = fake.date_time()
    records.append((address, city, phone_number, opened_at))

# Construct the sql with placeholder
statement = "INSERT INTO public.stores(address, city, phone_number, opened_at) VALUES (%s, %s, %s, %s)"

#Execute the SQL statement with the Values
cur.executemany(statement, records)

# Commit the transaction
conn.commit()
print("Stores Data Inserted Successfully!")

Stores Data Inserted Successfully!


TABLE CUSTOMERS

In [ ]:

# Number of records to insert
number_records = 1500

# Instantiate Faker
fake = Faker()
fake.unique.clear()

records = []

# Generate customer data
for _ in range(number_records):
    first_name = fake.first_name()
    last_name = fake.last_name()
    email = fake.unique.email()          # ✅ prevents duplicate email error
    phone_number = fake.msisdn()[:15]    # ✅ clean phone number
    created_at = fake.date_time_between(start_date='-2y', end_date='now')

    records.append(
        (first_name, last_name, email, phone_number, created_at)
    )

# SQL insert statement
statement = """
    INSERT INTO public.customers
    (first_name, last_name, email, phone_number, created_at)
    VALUES (%s, %s, %s, %s, %s)
"""

# Execute insert
try:
    cur.executemany(statement, records)
    conn.commit()
    print("✅ Customers data inserted successfully!")

except Exception as e:
    conn.rollback()   # Reset failed transaction
    print("❌ Error inserting data:", e)


❌ Error inserting data: duplicate key value violates unique constraint "customers_email_key"
DETAIL:  Key (email)=(robert51@example.com) already exists.



TABLE INGREDIENT

In [13]:
from faker import Faker
import random
import psycopg2

# Define number of records
number_records = 50

fake = Faker()
fake.unique.clear()

records = []

units = ["kg", "g", "litres", "ml", "pcs"]

# IMPORTANT: Reset failed transaction first
conn.rollback()

# Fetch existing ingredient names from database
cur.execute("SELECT name FROM public.ingredients;")
existing_names = {row[0] for row in cur.fetchall()}

# Generate new unique names that are NOT already in DB
while len(records) < number_records:
    name = fake.word()

    if name not in existing_names:
        stock_quantity = fake.random_int(min=1, max=500)
        unit = random.choice(units)

        records.append((name, stock_quantity, unit))
        existing_names.add(name)  # prevent duplicates in same batch

# SQL with conflict protection (VERY IMPORTANT)
statement = """
INSERT INTO public.ingredients (name, stock_quantity, unit)
VALUES (%s, %s, %s)
ON CONFLICT (name) DO NOTHING
"""

cur.executemany(statement, records)
conn.commit()

print("Ingredients Data Inserted Successfully!")


Ingredients Data Inserted Successfully!


TABLE ORDER

In [ ]:
number_records = 6000
records = []

# ---- RESET FAILED TRANSACTION ----
conn.rollback()

# ---- FETCH VALID FOREIGN KEYS ----
cur.execute("SELECT customer_id FROM customers;")
customer_ids = [row[0] for row in cur.fetchall()]

cur.execute("SELECT store_id FROM stores;")
store_ids = [row[0] for row in cur.fetchall()]

if not customer_ids or not store_ids:
    raise Exception("Customers or Stores table is empty.")

# ---- GENERATE ORDERS DATA ----
for _ in range(number_records):
    customer_id = fake.random_element(customer_ids)
    store_id = fake.random_element(store_ids)
    order_timestamp = fake.date_time()
    total_amount = round(fake.random.uniform(10, 200), 2)

    records.append((
        customer_id,
        store_id,
        order_timestamp,
        total_amount
    ))

# ---- INSERT DATA ----
statement = """
INSERT INTO public.orders (customer_id, store_id, order_timestamp, total_amount)
VALUES %s
"""

execute_values(cur, statement, records)
conn.commit()

print("Orders Data Inserted Successfully!")


Orders Data Inserted Successfully!


TABLE MENU_ITEM

In [17]:
# Number of records
number_records = 50

# Faker instance
fake = Faker()

records = []

# Possible categories
categories = ["Pizza", "Drink", "Dessert", "Side", "Pasta"]

# Possible sizes
sizes = ["Small", "Medium", "Large", "Extra Large"]

# Generate data
for _ in range(number_records):
    name = fake.word().capitalize()
    category = random.choice(categories)
    size = random.choice(sizes)

    records.append((name, category, size))

# SQL statement
statement = """
INSERT INTO public.menu_items (name, category, size)
VALUES (%s, %s, %s)
"""

# Execute
cur.executemany(statement, records)
conn.commit()

print("Menu Items Data Inserted Successfully!")

# Close connection
cur.close()
conn.close()

Menu Items Data Inserted Successfully!


TABLE ORDER_ITEM

In [ ]:
number_records = 10000
records = []

# ---- RESET FAILED TRANSACTION ----
conn.rollback()

# ---- FETCH VALID FOREIGN KEYS ----
cur.execute("SELECT order_id FROM orders;")
order_ids = [row[0] for row in cur.fetchall()]

cur.execute("SELECT item_id FROM menu_items;")
item_ids = [row[0] for row in cur.fetchall()]

if not order_ids or not item_ids:
    raise Exception("Orders or Menu_Items table is empty.")

# ---- GENERATE ORDER_ITEMS DATA ----
for _ in range(number_records):
    order_id = random.choice(order_ids)
    item_id = random.choice(item_ids)
    quantity = random.randint(1, 5)

    # Generate realistic price (between £5 and £50)
    price_each = round(random.uniform(5.00, 50.00), 2)

    records.append((
        order_id,
        item_id,
        quantity,
        price_each
    ))

# ---- INSERT DATA ----
statement = """
INSERT INTO public.order_items (order_id, item_id, quantity, price_each)
VALUES %s
"""

execute_values(cur, statement, records)
conn.commit()

print("Order_Items Data Inserted Successfully!")


Order_Items Data Inserted Successfully!


TABLE MENU_INGREDIENT

In [25]:
from faker import Faker
import random
from psycopg2.extras import execute_values

fake = Faker()
number_records = 5000
records = set()   # prevents duplicate (item_id, ingredient_id)

# ---- RESET FAILED TRANSACTION ----
conn.rollback()

# ---- FETCH VALID FOREIGN KEYS ----
cur.execute("SELECT item_id FROM menu_items;")
item_ids = [row[0] for row in cur.fetchall()]

cur.execute("SELECT ingredient_id FROM ingredients;")
ingredient_ids = [row[0] for row in cur.fetchall()]

if not item_ids or not ingredient_ids:
    raise Exception("menu_items or ingredients table is empty.")

# ---- GENERATE UNIQUE RELATIONSHIPS ----
while len(records) < number_records:
    item_id = random.choice(item_ids)
    ingredient_id = random.choice(ingredient_ids)

    quantity_required = round(random.uniform(0.05, 5.00), 2)

    records.add((
        item_id,
        ingredient_id,
        quantity_required
    ))

records = list(records)

# ---- INSERT DATA ----
statement = """
INSERT INTO public.menu_item_ingredients
(item_id, ingredient_id, quantity_required)
VALUES %s
ON CONFLICT (item_id, ingredient_id) DO NOTHING;
"""

execute_values(cur, statement, records)
conn.commit()

print("menu_item_ingredients Data Inserted Successfully!")


menu_item_ingredients Data Inserted Successfully!


TABLE STORE_INGREDIENT

In [ ]:
number_records = 3000
records = set()   # prevents duplicate (store_id, ingredient_id)

# ---- FETCH VALID FOREIGN KEYS ----
cur.execute("SELECT store_id FROM stores;")
store_ids = [row[0] for row in cur.fetchall()]

cur.execute("SELECT ingredient_id FROM ingredients;")
ingredient_ids = [row[0] for row in cur.fetchall()]

if not store_ids or not ingredient_ids:
    raise Exception("stores or ingredients table is empty.")

# ---- GENERATE UNIQUE RELATIONSHIPS ----
while len(records) < number_records:
    store_id = random.choice(store_ids)
    ingredient_id = random.choice(ingredient_ids)

    # realistic stock quantity (e.g., kg, litres, pcs etc.)
    stock_quantity = round(random.uniform(10, 500), 2)

    records.add((
        store_id,
        ingredient_id,
        stock_quantity
    ))

records = list(records)

# ---- INSERT DATA ----
statement = """
INSERT INTO public.store_ingredients
(store_id, ingredient_id, stock_quantity)
VALUES %s
ON CONFLICT (store_id, ingredient_id) DO NOTHING;
"""

execute_values(cur, statement, records)
conn.commit()

print("store_ingredients Data Inserted Successfully!")


store_ingredients Data Inserted Successfully!
